In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.decomposition import LatentDirichletAllocation

poi_matrix = np.zeros((40000, 85), dtype=np.float32)
directory = "cell_POIcat.csv"

for filename in os.listdir(directory):
    parts = filename.split(',')
    if len(parts) != 4:
        continue
    try:
        x, y, cat_id, count = map(int, parts)
        grid_idx = (x - 1) * 200 + (y - 1)
        poi_matrix[grid_idx, cat_id - 1] = count
    except ValueError:
        continue

poi_df = pd.DataFrame(poi_matrix)
print(f"Non-zero cells: {(poi_matrix.sum(axis=1) > 0).sum():,} / 40,000")

Non-zero cells: 20,146 / 40,000


In [ ]:
import ast, re

poi_cats = pd.read_csv(
    "POI_datacategories_with_top11.csv",
    header=None,
    names=["x", "y", "top_venues", "zone_label", "top_categories"]
)

# The top5_categories column looks like:
def parse_category_counts(raw):
    if pd.isna(raw):
        return {}
    result = {}
    for match in re.finditer(r'([^,(]+)\((\d+)\)', str(raw)):
        cat_name = match.group(1).strip()
        count = int(match.group(2))
        result[cat_name] = result.get(cat_name, 0) + count
    return result

poi_cats['cat_dict'] = poi_cats['top_categories'].apply(parse_category_counts)

# Get all unique top-level category names across the dataset
all_top_cats = sorted(set(
    cat for d in poi_cats['cat_dict'] for cat in d.keys()
))
print(f"Unique top-level categories found: {len(all_top_cats)}")
print(all_top_cats)

Unique top-level categories found: 11
['Arts and Entertainment', 'Business and Professional Services', 'Community and Government', 'Dining and Drinking', 'Event', 'Health and Medicine', 'Landmarks and Outdoors', 'Nightlife Spot', 'Retail', 'Sports and Recreation', 'Travel and Transportation']


In [3]:
# Run LDA on raw POI counts (no TF-IDF)
lda = LatentDirichletAllocation(
    n_components=5,
    random_state=42,
    learning_method='batch',
    max_iter=20
)
zone_distributions = lda.fit_transform(poi_df)

zone_cols = [f'zone_{i}_prob' for i in range(5)]
dist_df = pd.DataFrame(zone_distributions, columns=zone_cols)
poi_df = pd.concat([poi_df, dist_df], axis=1)
poi_df['grid_id'] = range(40000)
poi_df['dominant_zone'] = dist_df.values.argmax(axis=1)

# Print top POI categories per zone using REAL names (Cat_1..85)

feature_names = [f"Cat_{i}" for i in range(1, 86)]
for topic_idx, topic in enumerate(lda.components_):
    top_features = [feature_names[i] for i in topic.argsort()[:-6:-1]]
    print(f"Zone {topic_idx} Top anonymous POIs: {top_features}")

Zone 0 Top anonymous POIs: ['Cat_59', 'Cat_48', 'Cat_76', 'Cat_69', 'Cat_79']
Zone 1 Top anonymous POIs: ['Cat_4', 'Cat_40', 'Cat_48', 'Cat_13', 'Cat_69']
Zone 2 Top anonymous POIs: ['Cat_69', 'Cat_62', 'Cat_51', 'Cat_66', 'Cat_63']
Zone 3 Top anonymous POIs: ['Cat_81', 'Cat_84', 'Cat_75', 'Cat_63', 'Cat_79']
Zone 4 Top anonymous POIs: ['Cat_74', 'Cat_60', 'Cat_73', 'Cat_47', 'Cat_79']


In [4]:
# Build a grid_id → named category profile from the CSV
cat_profile = {}
for _, row in poi_cats.iterrows():
    try:
        x, y = int(row['x']), int(row['y'])
        grid_id = (x - 1) * 200 + (y - 1)
        cat_profile[grid_id] = {
            'zone_label': row['zone_label'],
            'cat_dict': row['cat_dict'],
            'top_venues': str(row['top_venues'])
        }
    except (ValueError, TypeError):
        continue

# For each LDA zone, find grid cells where that zone dominates
# then look up what named categories are most common there
zone_named_categories = {}
for zone_id in range(5):
    dominant_cells = poi_df[poi_df['dominant_zone'] == zone_id]['grid_id'].tolist()
    
    # Aggregate named category counts across all cells in this zone
    agg = {}
    for gid in dominant_cells:
        if gid in cat_profile:
            for cat, cnt in cat_profile[gid]['cat_dict'].items():
                agg[cat] = agg.get(cat, 0) + cnt
    
    # Sort and get top 5
    top = sorted(agg.items(), key=lambda x: -x[1])[:5]
    zone_named_categories[zone_id] = top
    print(f"\nZone {zone_id} — Top named categories:")
    for cat_name, cnt in top:
        print(f"  {cat_name}: {cnt:,}")
    
    # Also show the most common zone_labels from the CSV
    labels = [cat_profile[gid]['zone_label'] for gid in dominant_cells if gid in cat_profile]
    from collections import Counter
    label_counts = Counter(labels).most_common(3)
    print(f"  Most common CSV zone labels: {label_counts}")


Zone 0 — Top named categories:
  Business and Professional Services: 49,218
  Retail: 17,836
  Community and Government: 12,307
  Travel and Transportation: 9,913
  Dining and Drinking: 9,088
  Most common CSV zone labels: [('other', 2539), ('commercial', 300), ('food_leisure', 160)]

Zone 1 — Top named categories:
  Business and Professional Services: 22,487
  Dining and Drinking: 18,059
  Retail: 17,772
  Travel and Transportation: 6,034
  Community and Government: 5,342
  Most common CSV zone labels: [('other', 961), ('food_leisure', 422), ('commercial', 83)]

Zone 2 — Top named categories:
  Business and Professional Services: 61,489
  Retail: 37,384
  Community and Government: 29,126
  Dining and Drinking: 19,672
  Health and Medicine: 16,728
  Most common CSV zone labels: [('other', 4249), ('food_leisure', 409), ('commercial', 362)]

Zone 3 — Top named categories:
  Business and Professional Services: 23,306
  Retail: 12,403
  Community and Government: 7,091
  Travel and Transpo

In [ ]:
ZONE_NAMES = {
    0: "urban_services",      # Mixed offices, transit, retail — suburban connective tissue
    1: "food_entertainment",  # Dining-heavy, leisure, bars, restaurants
    2: "dense_urban_core",    # Highest density, all categories present, commercial hub
    3: "commercial_industrial", # Business/retail-heavy, low food, warehouses/offices
    4: "residential_civic"    # Community/govt leads, health, schools, elderly care
}

poi_df['zone_name'] = poi_df['dominant_zone'].map(ZONE_NAMES)

poi_df['csv_zone_label'] = poi_df['grid_id'].map(
    lambda gid: cat_profile.get(gid, {}).get('zone_label', 'unknown')
)

# Quick validation: cross-tab our LDA zones vs CSV zone labels
cross = pd.crosstab(
    poi_df['zone_name'],
    poi_df['csv_zone_label'],
    normalize='index'
).round(3)
print("LDA zone vs CSV label distribution (row-normalised):")
print(cross)

zone_features = poi_df[['grid_id', 'dominant_zone', 'zone_name'] + zone_cols + ['csv_zone_label']]
zone_features.to_parquet("grid_zone_features.parquet", index=False)

print("\nZone cell counts:")
print(poi_df['zone_name'].value_counts())

LDA zone vs CSV label distribution (row-normalised):
csv_zone_label         commercial  entertainment  food_leisure  \
zone_name                                                        
commercial_industrial       0.251          0.005         0.021   
dense_urban_core            0.072          0.002         0.081   
food_entertainment          0.054          0.012         0.277   
residential_civic           0.042          0.015         0.015   
urban_services              0.013          0.000         0.007   

csv_zone_label         mixed_food_commercial  other  unknown  
zone_name                                                     
commercial_industrial                  0.005  0.717    0.000  
dense_urban_core                       0.004  0.842    0.000  
food_entertainment                     0.028  0.630    0.000  
residential_civic                      0.001  0.926    0.000  
urban_services                         0.001  0.111    0.868  

Zone cell counts:
zone_name
urban_services